In [2]:
import pandas as pd

listings = pd.read_csv(
    "../outputs/clean_listings_with_rates.csv",
    low_memory=False
)

sold = pd.read_csv(
    "../outputs/clean_sold_with_rates.csv",
    low_memory=False
)

print("Listings:", listings.shape)
print("Sold:", sold.shape)

Listings: (616048, 60)
Sold: (439206, 65)


## Part 2: Convert Date Columns

To ensure accurate time-based analysis, all date-related fields are converted to the `datetime` data type.

Using a consistent datetime format allows us to:
- Perform date calculations and comparisons.
- Create time-based features.
- Validate the logical order of transactions in later steps.

In [3]:
date_columns = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

print("Listings date columns:")
for col in date_columns:
    if col in listings.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col}")

print("\nSold date columns:")
for col in date_columns:
    if col in sold.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col}")

Listings date columns:
✓ CloseDate
✓ PurchaseContractDate
✓ ListingContractDate
✓ ContractStatusChangeDate

Sold date columns:
✓ CloseDate
✓ PurchaseContractDate
✓ ListingContractDate
✓ ContractStatusChangeDate


In [4]:
for col in date_columns:
    if col in listings.columns:
        listings[col] = pd.to_datetime(
            listings[col],
            errors="coerce"
        )

    if col in sold.columns:
        sold[col] = pd.to_datetime(
            sold[col],
            errors="coerce"
        )

In [5]:
print("Listings date types:")
print(listings[date_columns].dtypes)

print("\nSold date types:")
print(sold[date_columns].dtypes)

Listings date types:
CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object

Sold date types:
CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object


## Part 3: Date Consistency Checks

After converting all date fields to the datetime format, we validate whether the transaction timeline follows the expected business process.

The expected order is:

ListingContractDate → PurchaseContractDate → CloseDate

Three boolean flags are created to identify records that violate this timeline.

In [6]:
print("Listings:")
print(listings.columns.tolist())

print("\nSold:")
print(sold.columns.tolist())

Listings:
['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'StateOrProvince', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool', 'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'HighSchool

In [7]:
sold["listing_after_close_flag"] = (
    sold["ListingContractDate"] > sold["CloseDate"]
)

In [9]:
sold["purchase_after_close_flag"] = (
    sold["PurchaseContractDate"] > sold["CloseDate"]
)

In [8]:
sold["negative_timeline_flag"] = (
    sold["PurchaseContractDate"] < sold["ListingContractDate"]
)

In [10]:
print("Listing after Close:")
print(sold["listing_after_close_flag"].sum())

print()

print("Purchase after Close:")
print(sold["purchase_after_close_flag"].sum())

print()

print("Negative Timeline:")
print(sold["negative_timeline_flag"].sum())

Listing after Close:
68

Purchase after Close:
252

Negative Timeline:
288


### Results

The timeline validation identified a small number of records with inconsistent transaction dates.

- **68 records** have a listing contract date later than the closing date.
- **252 records** have a purchase contract date later than the closing date.
- **288 records** have a purchase contract date earlier than the listing contract date.

These records were flagged for further review rather than removed because they may represent data entry errors, delayed updates, or special transaction cases.

In [11]:
listing_after_close = sold[
    sold["listing_after_close_flag"]
]

purchase_after_close = sold[
    sold["purchase_after_close_flag"]
]

negative_timeline = sold[
    sold["negative_timeline_flag"]
]

print(listing_after_close.head())
print(purchase_after_close.head())
print(negative_timeline.head())

      BuyerAgentAOR ListAgentAOR         Flooring ViewYN PoolPrivateYN  \
22         SanDiego     SanDiego              NaN  False         False   
83         SanDiego     SanDiego              NaN  False         False   
711        SanDiego     SanDiego              NaN   True         False   
11216           NaN          NaN  Carpet,Laminate   True         False   
11336           NaN          NaN              NaN   True         False   

       OriginalListPrice  ListingKey              ListAgentEmail  CloseDate  \
22             2195000.0  1059827487            RE@juliefeld.com 2024-01-30   
83             2705000.0  1059357897   lindsay@thedunlapteam.com 2024-01-25   
711            1600000.0  1054052014    joseph@arendsengroup.com 2024-01-01   
11216          1075000.0  1065738283  JulieBradenHomes@gmail.com 2024-03-29   
11336          1595000.0  1063548623      jamie@gregcummings.com 2024-03-20   

       ClosePrice  ... GarageSpaces HighSchoolDistrict  PostalCode  \
22      21

In [12]:
sold["date_issue_flag"] = (
    sold["listing_after_close_flag"] |
    sold["purchase_after_close_flag"] |
    sold["negative_timeline_flag"]
)

print("Total records with any date issue:",
      sold["date_issue_flag"].sum())

Total records with any date issue: 540


In [13]:
sold.to_csv(
    "../outputs/final_clean_sold.csv",
    index=False
)

listings.to_csv(
    "../outputs/final_clean_listings.csv",
    index=False
)